
## PCAを用いた炭素構造の次元圧縮による空間図示

### データ作成からデータ解析

炭素構造探索結果の説明変数をPCAで変換し次元圧縮して視覚的に意味のある変換になるのかを確かめる。


In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# データ取得
g_df_obs = pd.read_csv(
    "../data_calculated/Carbon8_descriptor_selected.csv", index_col=[0, 1])
g_df_all = pd.read_csv("../data_calculated/Carbon8_descriptor.csv", index_col=[0, 1])

g_descriptor_names = ['a0.25_rp1.0', 'a0.25_rp1.5', 'a0.25_rp2.0', 'a0.25_rp2.5',
                     'a0.25_rp3.0', 'a0.5_rp1.0', 'a0.5_rp1.5', 'a0.5_rp2.0', 'a0.5_rp2.5',
                     'a0.5_rp3.0', 'a1.0_rp1.0', 'a1.0_rp1.5', 'a1.0_rp2.0', 'a1.0_rp2.5',
                     'a1.0_rp3.0']


In [ ]:
def add_sptype(df):
    """nnatom2.0, 2.5, 3.0, 3.5, 4.0をnnatom_str sp, sp2_edge, sp2, sp2_tube, sp3に変換する。

    Args:
        df (pd.DataFrame): データ

    Returns:
        pd.DataFrame: nnatom_strカラムを追加したデータ
    """
    label_str = {2.0:"sp", 2.5: "sp2_edge", 3.0: "sp2", 
                 3.5: "sp2_tube", 4.0:"sp3"} 
    df["nnatom_str"] = [label_str[x] for x in df["nnatom"]]
    return df
g_df_obs = add_sptype(g_df_obs)
g_df_obs

In [ ]:
g_descriptor_names

寄与率を計算します。

In [ ]:
def pca_contribution(df, descriptor_names):
    """PCAの寄与率を計算して表示する。

    Args:
        df (pd.DataFrame): 観測データ
        descriptor_names ([str]): 説明変数リスト
    """
    Xraw = df.loc[:, descriptor_names].values
    # データ規格化
    scaler = StandardScaler()
    scaler.fit(Xraw)
    X = scaler.transform(Xraw)
    # データ解析
    # 次元圧縮
    explained_variance = []
    explained_variance_ratio = []
    ndim = Xraw.shape[1]
    print("ndim",ndim)
    pca = PCA(ndim)
    pca.fit(X)
    
    indx = [i for i in range(1,len(pca.explained_variance_ratio_)+1)]
    esum = [np.sum(pca.explained_variance_ratio_[:i+1]) for i in 
            range(len(pca.explained_variance_ratio_))]
    
    fig, ax = plt.subplots()
    ax.plot(indx,pca.explained_variance_ratio_,"o-", label="explained_variance_ratio")
    ax.plot(indx,esum,"o-", label="sum(explained_variance_ratio)")
    ax.legend()
    
pca_contribution(g_df_obs, g_descriptor_names)

以下はPCAで二次元空間へ変換しています。

In [ ]:
def apply_normalization(df, descriptor_names, scaler=None):
    """規格化を行う。

    scaler==Noneの場合はscalerをつくる。

    Args:
        df (pd.DataFrame): データ。 
        descriptor_names ([str]]): 説明変数名リスト
        scaler (StandardScaler, optional): StandardScalerインスタンス. Defaults to None.

    Returns:
        pd.DataFrame: データ
        [str]: 規格化を行った説明変数名リスト
        StandardScaler: StandardScaler
    """
    X_labels = []
    for s in descriptor_names:
        X_labels.append("s_{}".format(s))
    if X_labels[0] in df:
        return df, X_labels, scaler
    
    Xraw = df.loc[:, descriptor_names].values
    # データ規格化
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(Xraw)
    X = scaler.transform(Xraw)
    df[X_labels] = X
    return df, X_labels, scaler

def apply_pca(df, X_labels,  pca=None, ndim=2,): 
    """PCAを行う。

    Args:
        df (pd.DataFrame): データ
        X_labels ([str]): 説明変数名リスト
        pca (PCA, optional): PCAインスタンス. Defaults to None.
        ndim (int, optional): PCAの次元. Defaults to 2.

    Returns:
        [type]: [description]
    """
    label = []
    for x in range(ndim):
        label.append("pca{}".format(x+1))
    if label[0] in df:
        return df, label, pca
        
    X = df[X_labels].values
    # 2次元へ次元圧縮
    if pca is None:
        pca = PCA(ndim)
        pca.fit(X)
    X_pca = pca.transform(X)

    df_pca = pd.DataFrame(X_pca, columns=label)
    if "nnatom_str" in list(df.columns):
        df_pca["nnatom_str"] = df["nnatom_str"].values
    return df_pca, label, pca

g_df_obs, g_X_labels, g_scaler = apply_normalization(g_df_obs, g_descriptor_names)
g_df_all, _, _ = apply_normalization(g_df_all, g_descriptor_names, scaler=g_scaler)

g_df_pca, g_pca_labels, g_pca = apply_pca(g_df_obs, g_X_labels)
g_df_all_pca ,_,  _ = apply_pca(g_df_all, g_X_labels, pca=g_pca)

display(g_df_pca)

display(g_df_all_pca)


### 可視化 

各構造のメタ情報として局所sp配置名がついてます。この数字の意味は

- sp : sp配置
- sp2_edge : ナノリボンの端にありsp$^2$配置だがbondが一つ無い。
- sp2: sp$^2$配置
- sp2_tube: 細いナノチューブ中のsp$^2$
- sp3: sp$^3$配置

です。これを可視化します。

In [ ]:
# 可視化
def plot_Xpca(df_pca, label="nnatom_str"):
    """低次元化された説明変数の図示。

    Args:
        df_pca (pd.DataFrame)): データ
        label (str, optional): 低次元化された説明変数名のリスト. Defaults to "nnatom_str".
    """
    nnatom = df_pca[label]
    nnatom_uniq = np.unique(nnatom)
    print(nnatom_uniq)
    fig, ax = plt.subplots()
    for ic, nn in enumerate(nnatom_uniq):
        dfq = df_pca.query("{}=='{}'".format(label,nn))
        ax.scatter(dfq.loc[:,"pca1"],dfq.loc[:,"pca2"],  label=nn)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0,)
    ax.set_xlabel("pca1")
    ax.set_ylabel("pca2")
    
plot_Xpca(g_df_pca)

この説明変数には角度情報がありませんが、sp、bondが一つ無いsp$^2$、sp$^2$、sp$^3$と原子環境により綺麗に分かれているらしいことが分かります。


角度依存性がないRDFから評価していますが、sp2_edgeとspが分かれています。
全体のデータからするとどう見えるのでしょうか。


df_allを含めて可視化します。
等高線は分布の頻度が多い部分を描いてくれるkdeplotから来ています。
そこからはずれた点も多く見られます,
分布の頻度が多い部分はsp、sp$^2$、sp$^3$となっていることが分かります。

In [ ]:
def plot_X2d(df_all, df_select, label_str = "nnatom_str", alpha=0.1, show_kde=True, save_fig: bool=False):
    """plot explanatory variables in 2D

    Args:
        df_all (pd.DataFrame): all the explanatory variables in PCA
        df_select (pd.DataFrame): the selected descrptors in PCA
        label_str (str, optional): column name of nnatom_str. Defaults to "nnatom_str".
        alpha (float, optional): alpha channel value. Defaults to 0.1.
        show_kde (bool, optional): also show kde plot. Defaults to True.
    """
    fig, ax = plt.subplots(figsize=(5, 5))
    
    desc = ["pca1","pca2"]
    # 全体の表示
    if show_kde:
        sns.kdeplot(data=df_all , x=desc[0], y=desc[1], ax=ax)
    ax.scatter(df_all.loc[:, desc[0]].values, df_all.loc[:, desc[1]].values, 
                color="black", marker=".", alpha=alpha)    
        
    nnatom = df_select.loc[:,label_str]
    nnatom_uniq = np.unique(nnatom)
    for ic, nn in enumerate(nnatom_uniq):
        dfq = df_select.query("{}=='{}'".format(label_str,nn))
        ax.scatter(dfq.loc[:,desc[0]],dfq.loc[:,desc[1]],  label=nn)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0,)
    ax.set_xlabel(desc[0])
    ax.set_ylabel(desc[1])
    fig.tight_layout()
    if save_fig:
        import os
        os.makedirs("image_executed", exist_ok=True)
        fig.savefig("image_executed/PCA_carbon_atomic_environment.png")
    

plot_X2d(g_df_all_pca, g_df_pca, show_kde=True, save_fig=True)
plot_X2d(g_df_all_pca, g_df_pca, alpha=0.1, show_kde=False) # contourを消して表示する。


kdeplotの分布は人間の直感とは必ずしも一致しません。
これは図の描き方のせいでもあります。

角度依存性がないRDFから評価しているのでsp2_edgeとspは繋がって分布しているようです。